# Stage 4 — Model 4: K-Nearest Neighbors (`KNeighborsClassifier`)
**Project:** 10-Year Coronary Heart Disease (CHD) Risk Prediction (Framingham Heart Study)  
**Author:** Saif Asaad Naeem | Al-Maaqal University — Department of AI Engineering

---
### Notebook Roadmap:
1. **Environment Setup & Data Loading:** Load `framingham_balanced.csv` and apply the unified 80%/20% stratified split.
2. **Stage 1 — Baseline Model Evaluation:** Evaluate the default `KNeighborsClassifier (k=5, p=2)`.
3. **Stage 2 — Continuous Exploration via `RandomizedSearchCV ()`:** Sample neighbor counts and fractional Minkowski distance metrics $p \in [1.0, 3.0]$ using parentheses `()`.
4. **Stage 3 — Exhaustive Exploitation via `GridSearchCV []`:** Pinpoint the exact optimal neighborhood and distance metric ($k=2, p=1.81$) using square brackets `[]`.
5. **Final Results Comparison, Serialization & Diagnostic Plots:** Export `best_model4_knn.pkl` and generate diagnostic plots.

### 1. Environment Setup and Data Loading
Import required packages and initialize the 5-Fold Stratified Cross-Validation evaluator.

In [ ]:
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, uniform

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay

df = pd.read_csv('framingham_balanced.csv')
X = df.drop('TenYearCHD', axis=1).values
y = df['TenYearCHD'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

def evaluate_model(model, stage_name):
    cv_res = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'Stage': stage_name,
        'CV_Recall (%)': round(cv_res['test_recall'].mean() * 100, 2),
        'CV_Precision (%)': round(cv_res['test_precision'].mean() * 100, 2),
        'CV_F1 (%)': round(cv_res['test_f1'].mean() * 100, 2),
        'CV_ROC_AUC (%)': round(cv_res['test_roc_auc'].mean() * 100, 2),
        'CV_Accuracy (%)': round(cv_res['test_accuracy'].mean() * 100, 2),
        'Test_Recall (%)': round(recall_score(y_test, y_pred) * 100, 2),
        'Test_Precision (%)': round(precision_score(y_test, y_pred) * 100, 2),
        'Test_F1 (%)': round(f1_score(y_test, y_pred) * 100, 2),
        'Test_ROC_AUC (%)': round(roc_auc_score(y_test, y_prob) * 100, 2),
        'Test_Accuracy (%)': round(accuracy_score(y_test, y_pred) * 100, 2)
    }

print(f"Dataset Loaded: Train = {X_train.shape}, Test = {X_test.shape}")

### 2. Stage 1: Baseline `KNeighborsClassifier` Evaluation
Evaluate the untuned baseline KNN model.

In [ ]:
base_knn = KNeighborsClassifier(n_jobs=-1)
res_base = evaluate_model(base_knn, '1. Baseline KNN')
print("Baseline Evaluation:", res_base)

### 3. Stage 2: Broad Exploration via `RandomizedSearchCV ()`
Sample continuous Minkowski exponent $p$ and discrete $k$ and `leaf_size` distributions enclosed in parentheses `()`.

In [ ]:
param_dist_knn = {
    'n_neighbors': randint(2, 20),
    'p': uniform(1.0, 2.0),
    'leaf_size': randint(15, 55)
}

rand_knn = RandomizedSearchCV(
    estimator=KNeighborsClassifier(weights='distance', n_jobs=-1),
    param_distributions=param_dist_knn,
    n_iter=25,
    scoring='f1',
    cv=skf,
    random_state=42,
    n_jobs=-1
)
rand_knn.fit(X_train, y_train)
best_r = rand_knn.best_params_
print("Best Parameters from RandomizedSearchCV ():", best_r)
res_rand = evaluate_model(rand_knn.best_estimator_, '2. RandomizedSearchCV ()')

### 4. Stage 3: Fine-Grained Exploitation via `GridSearchCV []`
Bracket the optimal parameters inside explicit square-bracket lists `[]` for exhaustive local refinement.

In [ ]:
param_grid_knn = {
    'n_neighbors': sorted(list(set([max(1, best_r['n_neighbors'] - 2), max(1, best_r['n_neighbors'] - 1), best_r['n_neighbors'], best_r['n_neighbors'] + 1, best_r['n_neighbors'] + 2]))),
    'weights': ['distance', 'uniform'],
    'p': [round(max(1.0, best_r['p'] - 0.2), 2), round(best_r['p'], 2), round(best_r['p'] + 0.2, 2), 1.0, 2.0],
    'leaf_size': [max(10, best_r['leaf_size'] - 5), best_r['leaf_size'], best_r['leaf_size'] + 5]
}

grid_knn = GridSearchCV(
    estimator=KNeighborsClassifier(n_jobs=-1),
    param_grid=param_grid_knn,
    scoring='f1',
    cv=skf,
    n_jobs=-1
)
grid_knn.fit(X_train, y_train)
print("Best Parameters from GridSearchCV []:", grid_knn.best_params_)
res_grid = evaluate_model(grid_knn.best_estimator_, '3. GridSearchCV [] (Final)')

### 5. Final Results Comparison, Model Serialization & Diagnostic Plots
Summarize all three stages, export `best_model4_knn.pkl`, and plot the Confusion Matrix and ROC curve.

In [ ]:
results_df = pd.DataFrame([res_base, res_rand, res_grid])
print("\n=== Model 4 (KNeighborsClassifier) Optimization Progression Table ===")
print(results_df.to_string(index=False))

joblib.dump(grid_knn.best_estimator_, 'best_model4_knn.pkl')
print("\nSUMMARY_MODEL_4_KNN:", {'best_params': grid_knn.best_params_, 'metrics': res_grid})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
y_pred_final = grid_knn.best_estimator_.predict(X_test)
sns.heatmap(confusion_matrix(y_test, y_pred_final), annot=True, fmt='d', cmap='Purples', ax=axes[0])
axes[0].set_title('KNN Confusion Matrix (Test Set)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
RocCurveDisplay.from_estimator(grid_knn.best_estimator_, X_test, y_test, ax=axes[1], name='KNN (GridSearch)')
axes[1].set_title('KNN ROC Curve', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model4_knn.png', dpi=300)
plt.show()